In [171]:
import glfw
from OpenGL.GL import *
import OpenGL.GL.shaders
import numpy as np
import glm


In [172]:
glfw.init()
glfw.window_hint(glfw.VISIBLE, glfw.FALSE);
window = glfw.create_window(720, 600, "Programa", None, None)

if (window == None):
    print("Failed to create GLFW window")
    glfwTerminate()
    
glfw.make_context_current(window)

In [173]:
vertex_code = """
        attribute vec3 position;
        uniform mat4 mat_transformation;
        void main(){
            gl_Position = mat_transformation * vec4(position,1.0);
        }
        """

In [174]:
fragment_code = """
        uniform vec4 color;
        uniform int night;
        void main(){
            if(night == 0){
                gl_FragColor = color;
            }
            else{
                gl_FragColor = 0.5*color;
            }
            
        }
        """

In [175]:
# Request a program and shader slots from GPU
program  = glCreateProgram()
vertex   = glCreateShader(GL_VERTEX_SHADER)
fragment = glCreateShader(GL_FRAGMENT_SHADER)

# Set shaders source
glShaderSource(vertex, vertex_code)
glShaderSource(fragment, fragment_code)

# Compile shaders
glCompileShader(vertex)

if not glGetShaderiv(vertex, GL_COMPILE_STATUS):
    error = glGetShaderInfoLog(vertex).decode()
    print(error)
    raise RuntimeError("Erro de compilacao do Vertex Shader")

glCompileShader(fragment)
if not glGetShaderiv(fragment, GL_COMPILE_STATUS):
    error = glGetShaderInfoLog(fragment).decode()
    print(error)
    raise RuntimeError("Erro de compilacao do Fragment Shader")

# Attach shader objects to the program
glAttachShader(program, vertex)
glAttachShader(program, fragment)

# Build program
glLinkProgram(program)
if not glGetProgramiv(program, GL_LINK_STATUS):
    print(glGetProgramInfoLog(program))
    raise RuntimeError('Linking error')
    
# Make program the default program
glUseProgram(program)

# Request a buffer slot from GPU
buffer_VBO = glGenBuffers(1)
glBindBuffer(GL_ARRAY_BUFFER, buffer_VBO)




In [ ]:
# preparando espaço para 3 vértices usando 2 coordenadas (x,y)
vertices = np.zeros(0, [("position", np.float32, 3)])

def makecircle(num_vertices, radius):
    circvertices = np.zeros(num_vertices, [("position", np.float32, 3)])

    angle = 0.0
    for counter in range(num_vertices):
        angle += 2*np.pi/num_vertices 
        x = np.cos(angle)*radius
        y = np.sin(angle)*radius
        circvertices[counter]['position'] = [x,y,0.0]
    return circvertices

def Make_sol():
    global solStart, solLen, vertices
    solStart = len(vertices['position'])
    solVertices = makecircle(20, 0.12)
    solLen = len(solVertices['position'])
    vertices = np.concatenate((vertices, solVertices))

def Make_house():
    global houseStart, houseLen, vertices
    houseStart = len(vertices['position'])
    houseVertices = np.zeros(9, [("position", np.float32, 3)])
    houseVertices['position'] = [[0.0,0.0,0.0],
                                 [1.0,0.0,0.0],
                                 [1.0,1.0,0.0],
                                 [0.0,1.0,0.0],
                                 [1.0,1.0,0.0],
                                 [0.0,0.0,0.0],
                                 [-0.2,1.0,0.0],
                                 [1.2,1.0,0.0],
                                 [0.5,1.5,0.0]]

    houseLen = len(houseVertices['position'])
    vertices = np.concatenate((vertices, houseVertices))

def Make_Ground():
    global groundStart, groundLen, vertices
    groundStart = len(vertices['position'])
    groundVertices = np.zeros(4, [("position", np.float32, 3)])
    groundVertices['position'] = [[-5.0,0.0,0.0],
                                 [5.0,0.0,0.0],
                                 [-5.0,-5.0,0.0],
                                 [5.0,-5.0,0.0]]

    groundLen = len(groundVertices['position'])
    vertices = np.concatenate((vertices, groundVertices))

def Make_Cloud():
    global cloudStart, cloudLen, vertices
    cloudStart = len(vertices['position'])
    
    # 3 círculos para formar a nuvem
    c1 = makecircle(20, 0.10)
    # Circulo 2 para baixo e para a esquerda
    c2 = makecircle(20, 0.06)
    c2['position'][:, 0] -= 0.12
    c2['position'][:, 1] -= 0.02
    # Circulo 3 para baixo e para a direita
    c3 = makecircle(20, 0.06)
    c3['position'][:, 0] += 0.12
    c3['position'][:, 1] -= 0.02
    
    # Junta os três círculos num array só
    cloudVertices = np.concatenate((c1, c2, c3))
    
    # Todos mesmo tamanho
    cloudLen = len(c1) 
    vertices = np.concatenate((vertices, cloudVertices))
    
def Make_Tree():
    global treeStart, treeLen, vertices
    treeStart = len(vertices['position'])
    
    # Tronco: retangulo de 2 triangulos (6 vértices)
    tronco = np.zeros(6, [("position", np.float32, 3)])
    tronco['position'] = [
        [-0.05, 0.0, 0.0], [0.05, 0.0, 0.0], [0.05, 0.2, 0.0],
        [-0.05, 0.0, 0.0], [0.05, 0.2, 0.0], [-0.05, 0.2, 0.0]
    ]
    
    # Folhas: triangulo grande em cima do tronco (3 vértices)
    folhas = np.zeros(3, [("position", np.float32, 3)])
    folhas['position'] = [
        [-0.2, 0.1, 0.0],
        [ 0.2, 0.1, 0.0],
        [ 0.0, 0.4, 0.0]  
    ]
                                 
    treeVertices = np.concatenate((tronco, folhas))
    treeLen = len(treeVertices['position'])
    vertices = np.concatenate((vertices, treeVertices))

Make_sol()
Make_house()
Make_Ground()
Make_Cloud()
Make_Tree()
vertices

array([([ 1.1412678e-01,  3.7082039e-02,  0.0000000e+00],),
       ([ 9.7082041e-02,  7.0534229e-02,  0.0000000e+00],),
       ([ 7.0534229e-02,  9.7082041e-02,  0.0000000e+00],),
       ([ 3.7082039e-02,  1.1412678e-01,  0.0000000e+00],),
       ([ 7.3478810e-18,  1.2000000e-01,  0.0000000e+00],),
       ([-3.7082039e-02,  1.1412678e-01,  0.0000000e+00],),
       ([-7.0534229e-02,  9.7082041e-02,  0.0000000e+00],),
       ([-9.7082041e-02,  7.0534229e-02,  0.0000000e+00],),
       ([-1.1412678e-01,  3.7082039e-02,  0.0000000e+00],),
       ([-1.2000000e-01,  1.4695762e-17,  0.0000000e+00],),
       ([-1.1412678e-01, -3.7082039e-02,  0.0000000e+00],),
       ([-9.7082041e-02, -7.0534229e-02,  0.0000000e+00],),
       ([-7.0534229e-02, -9.7082041e-02,  0.0000000e+00],),
       ([-3.7082039e-02, -1.1412678e-01,  0.0000000e+00],),
       ([-2.2043642e-17, -1.2000000e-01,  0.0000000e+00],),
       ([ 3.7082039e-02, -1.1412678e-01,  0.0000000e+00],),
       ([ 7.0534229e-02, -9.7082041e-02,

In [ ]:
def drawSol():
    angle = day_cycle
    mat_rotation       = np.array([    [np.cos(angle), -np.sin(angle), 0.0, 0.0], 
                                    [np.sin(angle), np.cos(angle), 0.0, 0.0], 
                                    [0.0, 0.0, 1.0, 0.0], 
                                    [0.0, 0.0, 0.0, 1.0]], np.float32)
    
    mat_translation = np.array([    [1.0, 0.0, 0.0, 0.8], 
                                    [0.0, 1.0, 0.0, 0.0], 
                                    [0.0, 0.0, 1.0, 0.0], 
                                    [0.0, 0.0, 0.0, 1.0]], np.float32)
    
    mat_final = mat_rotation @ mat_translation
    
    loc = glGetUniformLocation(program, "mat_transformation")
    glUniformMatrix4fv(loc, 1, GL_TRUE, mat_final)

  
    glUniform4f(loc_color, 1.0, 1.0, 0.0, 1.0) ### modificando a cor do objeto!
    glDrawArrays(GL_TRIANGLE_FAN, solStart, solLen)


def drawLuna():
    angle = day_cycle + np.pi
    mat_rotation       = np.array([    [np.cos(angle), -np.sin(angle), 0.0, 0.0], 
                                    [np.sin(angle), np.cos(angle), 0.0, 0.0], 
                                    [0.0, 0.0, 1.0, 0.0], 
                                    [0.0, 0.0, 0.0, 1.0]], np.float32)
    
    mat_translation = np.array([    [1.0, 0.0, 0.0, 0.8], 
                                    [0.0, 1.0, 0.0, 0.0], 
                                    [0.0, 0.0, 1.0, 0.0], 
                                    [0.0, 0.0, 0.0, 1.0]], np.float32)
    
    mat_final = mat_rotation @ mat_translation
    
    loc = glGetUniformLocation(program, "mat_transformation")
    glUniformMatrix4fv(loc, 1, GL_TRUE, mat_final)

  
    glUniform4f(loc_color, 1.95, 1.95, 1.90, 1.0) ### modificando a cor do objeto!
    glDrawArrays(GL_TRIANGLE_FAN, solStart, solLen)



def drawHouse():

    mat_scale      = np.array([    [0.2, 0.0, 0.0, 0.0], 
                                    [0.0, 0.2, 0.0, 0.0], 
                                    [0.0, 0.0, 1.0, 0.0], 
                                    [0.0, 0.0, 0.0, 1.0]], np.float32)
    
    mat_translation = np.array([    [1.0, 0.0, 0.0, -0.5], 
                                    [0.0, 1.0, 0.0, 0.0], 
                                    [0.0, 0.0, 1.0, 0.0], 
                                    [0.0, 0.0, 0.0, 1.0]], np.float32)
    
    mat_final = mat_scale @ mat_translation
    
    loc = glGetUniformLocation(program, "mat_transformation")
    glUniformMatrix4fv(loc, 1, GL_TRUE, mat_final)

  
    glUniform4f(loc_color, 0.95, 0.95, 0.90, 1.0) ### modificando a cor do objeto!
    glDrawArrays(GL_TRIANGLES, houseStart, houseLen-3)

    glUniform4f(loc_color, 0.90, 0.0, 0.10, 1.0) ### modificando a cor do objeto!
    glDrawArrays(GL_TRIANGLES, houseStart+6, houseLen-6)


def drawShadow():
    if night == 1.0:
        return
    
    # ciclo = 0 (nascer do sol); ciclo = pi/2 (meio dia); ciclo = pi (por do sol)
    sc = -np.cos(day_cycle)*2.0
    mat_scale      = np.array([    [-0.2, 0.0, 0.0, 0.0], 
                                    [0.0, -0.2, 0.0, 0.0], 
                                    [0.0, 0.0, 1.0, 0.0], 
                                    [0.0, 0.0, 0.0, 1.0]], np.float32)
    
    mat_shearing = np.array([    [1.0, -sc, 0.0, 0.0], 
                                    [0.0, 1.0, 0.0, 0.0], 
                                    [0.0, 0.0, 1.0, 0.0], 
                                    [0.0, 0.0, 0.0, 1.0]], np.float32)
    
    mat_translation = np.array([    [1.0, 0.0, 0.0, -0.5], 
                                    [0.0, 1.0, 0.0, 0.0], 
                                    [0.0, 0.0, 1.0, 0.0], 
                                    [0.0, 0.0, 0.0, 1.0]], np.float32)
    
    mat_final = mat_translation @ mat_shearing
    mat_final = mat_scale @ mat_final
    
    loc = glGetUniformLocation(program, "mat_transformation")
    glUniformMatrix4fv(loc, 1, GL_TRUE, mat_final)

  
    glUniform4f(loc_color, 0.0, 0.0, 0.0, 1.0) ### modificando a cor do objeto!
    glDrawArrays(GL_TRIANGLES, houseStart, houseLen)


def drawGround():

    mat_final = np.array([    [1.0, 0.0, 0.0, 0.0], 
                                    [0.0, 1.0, 0.0, 0.0], 
                                    [0.0, 0.0, 1.0, 0.0], 
                                    [0.0, 0.0, 0.0, 1.0]], np.float32)
    

    loc = glGetUniformLocation(program, "mat_transformation")
    glUniformMatrix4fv(loc, 1, GL_TRUE, mat_final)

  
    glUniform4f(loc_color, 0.1, 0.75, 0.0, 1.0) ### modificando a cor do objeto!
    glDrawArrays(GL_TRIANGLES, groundStart, groundLen)

def drawCloud():
    if night == 1.0:
        return 
        
    # Quando day_cycle é 0, tx = 1.0
    # Quando day_cycle é pi, tx = -1.0
    tx = 1.0 - (day_cycle / np.pi)*2.0
    
    mat_translation = np.array([    [1.0, 0.0, 0.0, tx], 
                                    [0.0, 1.0, 0.0, 0.6], 
                                    [0.0, 0.0, 1.0, 0.0], 
                                    [0.0, 0.0, 0.0, 1.0]], np.float32)
    
    loc = glGetUniformLocation(program, "mat_transformation")
    glUniformMatrix4fv(loc, 1, GL_TRUE, mat_translation)

    glUniform4f(loc_color, 1.0, 1.0, 1.0, 1.0) 
    
    # Desenha os 3 círculos da nuvem
    glDrawArrays(GL_TRIANGLE_FAN, cloudStart, cloudLen)                  
    glDrawArrays(GL_TRIANGLE_FAN, cloudStart + cloudLen, cloudLen)       
    glDrawArrays(GL_TRIANGLE_FAN, cloudStart + 2*cloudLen, cloudLen)

def drawTree():
    sy = 0.5 + total_time*0.2
    sx = 0.5 + total_time*0.15
    
    mat_scale = np.array([          [sx, 0.0, 0.0, 0.0], 
                                    [0.0, sy, 0.0, 0.0], 
                                    [0.0, 0.0, 1.0, 0.0], 
                                    [0.0, 0.0, 0.0, 1.0]], np.float32)
    
    mat_translation = np.array([    [1.0, 0.0, 0.0, -0.6], 
                                    [0.0, 1.0, 0.0, -0.5], 
                                    [0.0, 0.0, 1.0, 0.0], 
                                    [0.0, 0.0, 0.0, 1.0]], np.float32)
    
    mat_final = mat_translation @ mat_scale
    
    loc = glGetUniformLocation(program, "mat_transformation")
    glUniformMatrix4fv(loc, 1, GL_TRUE, mat_final)

    # Tronco marrom
    glUniform4f(loc_color, 0.55, 0.27, 0.07, 1.0)
    glDrawArrays(GL_TRIANGLES, treeStart, 6)

    # Folhas verdes
    glUniform4f(loc_color, 0.0, 0.6, 0.2, 1.0)
    glDrawArrays(GL_TRIANGLES, treeStart + 6, 3)

In [178]:
glBufferData(GL_ARRAY_BUFFER, vertices['position'].nbytes, vertices['position'], GL_DYNAMIC_DRAW)
glBindBuffer(GL_ARRAY_BUFFER, buffer_VBO)

# Bind the position attribute
# --------------------------------------
stride = vertices.strides[0]
offset = ctypes.c_void_p(0)


loc = glGetAttribLocation(program, "position")
glEnableVertexAttribArray(loc)
glVertexAttribPointer(loc, 3, GL_FLOAT, False, stride, offset)

loc_color = glGetUniformLocation(program, "color")

R = 0.7
G = 0.0
B = 0.2



In [ ]:
# exemplo para matriz de escala

def key_event(window,key,scancode,action,mods):
    global total_time, day_cycle, night
    
    # Seta esquerda avança o tempo
    if key == 263: 
         total_time += 0.01
    # Seta direita volta o tempo
    if key == 262: 
         total_time -= 0.01

    # Evita árvore escalando negativa
    if total_time < 0.0:
        total_time = 0.0

    day_cycle = total_time % (2*np.pi)

    if day_cycle > np.pi:
        night = 1.0
    else:
        night = 0.0

    loc_night = glGetUniformLocation(program, "night")
    glUniform1i(loc_night, int(night))
    
glfw.set_key_callback(window,key_event)


In [180]:
glfw.show_window(window)
total_time = 0.0
day_cycle = 0.0
night = 0.0
while not glfw.window_should_close(window):


    if (night == 0.0):
        glClearColor(173.0/255, 216.0/255, 230.0/255, 1.0)
        
    else:   
        glClearColor(0.1, 0.1, 0.5, 1.0)

    glClear(GL_COLOR_BUFFER_BIT) 
    
    drawSol()
    drawLuna()
    drawCloud()
    drawGround()
    drawHouse()
    drawShadow()
    drawTree()
    
    glfw.swap_buffers(window)
    glfw.poll_events() 

glfw.terminate()